This is a file to observe and archive existing data on the C: drive version of NCA_DATA

In [11]:
from pathlib import Path
import shutil
import pandas as pd

SOURCE = Path(r"C:\NCA_DATA")   # working copy
ARCHIVE = Path(r"A:\NCA_DATA")  # archive


def build_file_index(root):
    """
    Return a dictionary:
        relative_path -> Path object

    Files only. Directories themselves are not treated as objects to sync.
    """
    return {
        path.relative_to(root): path
        for path in root.rglob("*")
        if path.is_file()
    }


source_files = build_file_index(SOURCE)
archive_files = build_file_index(ARCHIVE)

all_relative_paths = sorted(set(source_files) | set(archive_files))

records = []

for rel in all_relative_paths:
    src = source_files.get(rel)
    arc = archive_files.get(rel)

    if src is not None and arc is None:
        status = "ONLY_C"
    elif src is None and arc is not None:
        status = "ONLY_A"
    else:
        # Same relative path exists in both places.
        # We are only comparing size here for auditing;
        # nothing will be overwritten regardless.
        if src.stat().st_size == arc.stat().st_size:
            status = "BOTH_SAME_SIZE"
        else:
            status = "BOTH_DIFFERENT_SIZE"

    records.append({
        "relative_path": str(rel),
        "status": status,
        "c_size_bytes": src.stat().st_size if src else None,
        "a_size_bytes": arc.stat().st_size if arc else None,
    })

audit = pd.DataFrame(records)

audit.to_csv(
    r"C:\NCA_DATA\directory_audit.csv",
    index=False
)

print(audit["status"].value_counts())

status
ONLY_A                 13263
BOTH_SAME_SIZE          7625
ONLY_C                  3430
BOTH_DIFFERENT_SIZE        1
Name: count, dtype: int64


In [12]:
to_copy_c = audit[audit["status"] == "ONLY_C"].copy()

print(f"Files that exist on C but not A: {len(to_copy_c):,}")

to_copy_c[["relative_path", "c_size_bytes"]]

total_bytes = to_copy_c["c_size_bytes"].sum()

print(f"{total_bytes / 1024**3:.2f} GiB")

Files that exist on C but not A: 3,430
1.92 GiB


In [13]:
copied = []
skipped = []
errors = []

for rel_string in to_copy["relative_path"]:
    rel = Path(rel_string)

    src = SOURCE / rel
    dst = ARCHIVE / rel

    try:
        # Critical safety check:
        # NEVER overwrite an existing file.
        if dst.exists():
            skipped.append({
                "relative_path": str(rel),
                "reason": "destination already exists"
            })
            continue

        # Create only the parent directory required for this file.
        dst.parent.mkdir(parents=True, exist_ok=True)

        # copy2 preserves timestamps and basic metadata.
        shutil.copy2(src, dst)

        copied.append(str(rel))

    except Exception as e:
        errors.append({
            "relative_path": str(rel),
            "error": repr(e)
        })

print(f"Copied: {len(copied)}")
print(f"Skipped: {len(skipped)}")
print(f"Errors: {len(errors)}")

Copied: 0
Skipped: 1130
Errors: 0
